# 第7章章节实践：HCCL 通信扩展性

## 本节学习目标

本实践要求综合运用本章知识完成可复现的工程任务。请保留命令、参数、正确性结果和分析结论。

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
%%bash
set -e
command -v cmake
command -v npu-smi
npu-smi info
printf "ASCEND_HOME_PATH=%s\n" "${ASCEND_HOME_PATH:?请先 source CANN set_env.sh}"


## 必要背景与实验材料

本实践基于本章 `src/` 中的课程工程副本。开始前应完成前面各小节，并能解释工程的关键源码、构建入口、正确性门槛和计时字段。

## 实践任务

1. 确认真实 HCCL backend 和 rank/device 映射
2. 选择环境允许的多个 rank 数运行同一矩阵
3. 记录 collective、transfer、local_spmv_launch_to_complete_ms（本地 Ascend C Device SpMV launch-to-complete）、total 和 error
4. 解释最优 rank 数以及本地 Device SpMV 与 collective 同步的组织开销

## 核心知识与关键源码解析

本章实践只调整 Notebook 中的运行参数和实验配置，不修改 `src/`。请保持输入与正确性阈值一致，每次只改变一个变量，并说明它如何影响数据流和性能。

## 实验记录模板与通信配置

先执行 `hostname -I` 查看本机 IPv4，并执行 `for d in 0 1 2 3; do hccn_tool -i "$d" -ip -g; done` 查询 Device IP。需要 rank table 时，按设备顺序设置 `SERVER_IP`、`DEVICE_IPS_2` 和 `DEVICE_IPS_4`，例如：

```bash
export SERVER_IP="<本机实际服务端IP>"
export DEVICE_IPS_2="<device0_ip> <device1_ip>"
export DEVICE_IPS_4="<device0_ip> <device1_ip> <device2_ip> <device3_ip>"
```

三个变量均未设置时，下面的 Cell 使用工程已有的单机 root-info 初始化，不要求 rank table；只设置其中一部分时会忽略不完整配置并使用 root-info，避免环境中的残留变量阻断实验。

In [ ]:
%%bash
set -euo pipefail
cd src/hccl_spmv
HCCL_SPMV_REQUIRE_REAL=1 bash scripts/build.sh

configured=0
[[ -n "${SERVER_IP:-}" ]] && configured=$((configured + 1))
[[ -n "${DEVICE_IPS_2:-}" ]] && configured=$((configured + 1))
[[ -n "${DEVICE_IPS_4:-}" ]] && configured=$((configured + 1))

if [[ "$configured" -eq 3 ]]; then
  read -r -a device_ips_2 <<< "$DEVICE_IPS_2"
  read -r -a device_ips_4 <<< "$DEVICE_IPS_4"
  [[ "${#device_ips_2[@]}" -eq 2 ]] || { echo "错误：DEVICE_IPS_2 必须恰好包含 2 个 IP" >&2; exit 2; }
  [[ "${#device_ips_4[@]}" -eq 4 ]] || { echo "错误：DEVICE_IPS_4 必须恰好包含 4 个 IP" >&2; exit 2; }
  python3 scripts/generate_rank_table.py --server-ip "$SERVER_IP" --device-ip "${device_ips_2[@]}" --output rank_table_2p.json
  python3 scripts/generate_rank_table.py --server-ip "$SERVER_IP" --device-ip "${device_ips_4[@]}" --output rank_table_4p.json
  bash scripts/run_scaling.sh --rank-table rank_table_2p.json --matrix U1 --npus-list 2 --warmup 3 --repeat 10
  bash scripts/run_scaling.sh --rank-table rank_table_4p.json --matrix U1 --npus-list 4 --warmup 3 --repeat 10
else
  if [[ "$configured" -gt 0 ]]; then
    echo "警告：rank-table IP 变量未设置完整，忽略不完整配置。" >&2
  fi
  echo "使用单机 root-info 初始化运行 2/4 rank。"
  bash scripts/run_scaling.sh --matrix U1 --npus-list 2,4 --warmup 3 --repeat 10
fi


## 评价标准

报告必须明确 backend 边界；collective/transfer 用于通信与搬移结论，local SpMV（Ascend C RTC Device kernel）用 local_spmv_launch_to_complete_ms 字段报告并说明其计时边界。

## 查看参考答案

参考答案给出方法和判断依据，不提供虚构的固定性能数字。

## 预期现象与结果分析

正确性门槛应首先通过；性能结果随硬件、软件栈和系统负载变化。若修改后没有加速或出现退化，也应依据阶段计时、通信次数或资源竞争给出解释。

## 实践小结

完成报告时，应明确实验环境、唯一修改变量、正确性门槛、计时口径和观察到的限制。

## 工程实践提交物与完成标准

章测必须基于 `src/hccl_spmv/`，不得只回答概念题。操作链：生成 rank table → 确认 real backend → 建 communicator → 运行 Broadcast/AllGather → 改 2/4/8 rank → 查 rank 日志 → 记录通信/端到端时间；local SpMV 必须由 Ascend C RTC Kernel 执行。

提交物：实际命令与环境；阅读或修改的真实文件/函数/参数；字段为“Ranks、HCCL、Transfer、Local NPU SpMV、Total、Error、日志状态”的结果表；正确性判据；基于数据的结论。性能数字不作为固定答案。

完成标准：命令指向真实脚本或可执行文件，数据来自同口径运行，并能解释结果。


## 四类考核

以下四题中，客观题答案唯一，凭 `src/hccl_spmv/scripts/generate_rank_table.py` 即可判定；简单/中等/困难题基于本章实验，要求用命令、输出、CSV 或计算过程作为证据，不接受无证据的概念回答。

### 1. 客观题

（1）单选：`generate_rank_table.py` 只接受下列哪一组命令行参数（　）

A. `--server-ip`、`--device-ip`、`--output`

B. `--server-ip`、`--device-id`、`--output`

C. `--rank-table`、`--npus-list`、`--device-ip`

D. `--server-ip`、`--rank-id`、`--device-id`

（2）判断（对/错）：`generate_rank_table.py` 按 `--device-ip` 的地址顺序自动生成 `device_id` 与 `rank_id`，并拒绝重复的 `device_ip`；2 rank 与 4 rank 实验必须分别使用各自匹配的 rank table 运行（不能用一个 4-rank JSON 配合 `--npus-list 2,4`）。（　）

### 2. 简单题

给出 2 rank 运行命令与输出：Actual Compute Backend、Communication Backend、Correctness、Error，以及 `local_spmv_launch_to_complete_ms` 的值；解释该字段的计时边界（本地 Ascend C SpMV 从 launch 到 complete）。

### 3. 中等题

描述 Broadcast → 本地 Ascend C Device SpMV → AllGather 的数据流，说明每次 solve 中 collective 的次数与字段对应关系；给出一次运行的 HCCL Communication、Data Transfer、Synchronization 字段值作为证据。

### 4. 困难题

用 2 与 4 rank 实测结果诊断扩展性：把 total 分解为 collective、transfer、本地 SpMV、同步，计算每 rank 通信量与并行效率，说明最优 rank 数；再构造一个失败场景（如 rank table 中 device/ip 重复），预测日志错误并说明如何用 rank 日志定位。
